In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from torchvision import transforms
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import random
import shutil
import gc
import zipfile
import time

# ─────────────────────────────────────────
# 1. GPU Setup
# ─────────────────────────────────────────
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

DEVICE = "cuda:0"

# ─────────────────────────────────────────
# 2. HuggingFace Token
#    Stored in Kaggle Secrets — never hardcoded
#    Add via: Kaggle → Settings → Secrets → HF_TOKEN
# ─────────────────────────────────────────
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"]               = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ HuggingFace login successful.")
except Exception as e:
    print(f"⚠️  HF token not found in Kaggle Secrets: {e}")
    print("   Proceeding without token — may be slower.")
    HF_TOKEN = None

# ─────────────────────────────────────────
# 3. Disk Space Helper
# ─────────────────────────────────────────
def get_dir_size_gb(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / 1e9

# ─────────────────────────────────────────
# 4. Paths
# ─────────────────────────────────────────
raw_dataset_dir  = '/kaggle/input/datasets/hades199/vr-final-project-dataset/vr-final-project-dataset'
partition_file   = os.path.join(raw_dataset_dir, 'Eval/list_eval_partition.txt')
bbox_file        = os.path.join(raw_dataset_dir, 'Anno/list_bbox_inshop.txt')
cropped_dir      = '/kaggle/input/datasets/hades199/3c-yolo-cropped-images'
temp_dir         = '/kaggle/working/clip_temp'
final_zip        = '/kaggle/working/clip_finetuned_v2'
CLIP_CACHE       = '/kaggle/working/cache/clip_base'

os.makedirs(temp_dir,   exist_ok=True)
os.makedirs(CLIP_CACHE, exist_ok=True)

# ─────────────────────────────────────────
# 5. Download CLIP once → cache → offline
# ─────────────────────────────────────────
os.environ["TRANSFORMERS_OFFLINE"] = "0"

if not os.path.exists(os.path.join(CLIP_CACHE, 'config.json')):
    print("\nDownloading base CLIP (one time only)...")
    kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
    m = CLIPModel.from_pretrained(
        "openai/clip-vit-base-patch32", **kwargs
    )
    p = CLIPProcessor.from_pretrained(
        "openai/clip-vit-base-patch32", **kwargs
    )
    m.save_pretrained(CLIP_CACHE)
    p.save_pretrained(CLIP_CACHE)
    del m
    torch.cuda.empty_cache()
    print(f"✅ CLIP cached at {CLIP_CACHE}")
else:
    print(f"\n✅ CLIP already cached at {CLIP_CACHE}")

# Force offline — no more network calls
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("🔒 Offline mode ON — loading everything from disk.\n")

# ─────────────────────────────────────────
# 6. Load Partition + Item IDs
# ─────────────────────────────────────────
print("Loading dataset metadata...")

df_part = pd.read_csv(partition_file, sep=r'\s+', skiprows=1)
df_bbox = pd.read_csv(bbox_file,      sep=r'\s+', skiprows=1)
df      = pd.merge(df_part, df_bbox, on='image_name')

df_train = df[df['evaluation_status'] == 'train'].reset_index(drop=True)

def extract_item_id(image_name):
    parts = image_name.replace('\\', '/').split('/')
    for part in parts:
        if part.startswith('id_'):
            return part
    return None

df_train['item_id'] = df_train['image_name'].apply(extract_item_id)
df_train = df_train.dropna(subset=['item_id'])

id_counts = df_train['item_id'].value_counts()
valid_ids  = id_counts[id_counts >= 2].index
df_train   = df_train[df_train['item_id'].isin(valid_ids)].reset_index(drop=True)

print(f"Train images   : {len(df_train)}")
print(f"Unique item IDs: {df_train['item_id'].nunique()}")

# ─────────────────────────────────────────
# 7. Augmentation Pipeline
# ─────────────────────────────────────────
augment = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.1
        )
    ], p=0.5),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3)
    ], p=0.2),
])

# ─────────────────────────────────────────
# 8. Dataset
# ─────────────────────────────────────────
class DeepFashionContrastiveDataset(Dataset):
    def __init__(self, df, cropped_dir, processor):
        self.processor   = processor
        self.cropped_dir = cropped_dir

        self.groups = {}
        for _, row in df.iterrows():
            iid      = row['item_id']
            rel_path = row['image_name'].replace('/', '_')
            p1 = os.path.join(cropped_dir, row['image_name'])
            p2 = os.path.join(cropped_dir, rel_path)
            path = p1 if os.path.exists(p1) else \
                   (p2 if os.path.exists(p2) else None)
            if path:
                self.groups.setdefault(iid, []).append(path)

        self.groups   = {k: v for k, v in self.groups.items() if len(v) >= 2}
        self.item_ids = list(self.groups.keys())
        print(f"Valid item IDs with ≥2 images: {len(self.item_ids)}")

    def __len__(self):
        return len(self.item_ids)

    def __getitem__(self, idx):
        item_id = self.item_ids[idx]
        paths   = self.groups[item_id]
        p1, p2  = random.sample(paths, 2)

        try:
            img1 = Image.open(p1).convert("RGB")
        except:
            img1 = Image.new("RGB", (224, 224))

        try:
            img2 = Image.open(p2).convert("RGB")
        except:
            img2 = Image.new("RGB", (224, 224))

        img1 = augment(img1)
        img2 = augment(img2)

        proc1 = self.processor(images=img1, return_tensors="pt")
        proc2 = self.processor(images=img2, return_tensors="pt")

        return {
            "pixel_values_1": proc1["pixel_values"].squeeze(0),
            "pixel_values_2": proc2["pixel_values"].squeeze(0),
            "item_id"        : item_id
        }

# ─────────────────────────────────────────
# 9. NT-Xent Loss
# ─────────────────────────────────────────
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1, z2):
        N   = z1.size(0)
        z   = torch.cat([z1, z2], dim=0)
        sim = torch.mm(z, z.T) / self.temperature

        mask = torch.eye(2 * N, device=z.device).bool()
        sim.masked_fill_(mask, float('-inf'))

        labels = torch.cat([
            torch.arange(N, 2 * N),
            torch.arange(0, N)
        ]).to(z.device)

        return F.cross_entropy(sim, labels)

# ─────────────────────────────────────────
# 10. Vision Features Helper
# ─────────────────────────────────────────
def get_vision_features(model, pixel_values):
    m              = model.module if isinstance(model, nn.DataParallel) else model
    vision_outputs = m.vision_model(pixel_values=pixel_values)
    pooled_output  = vision_outputs.pooler_output
    return m.visual_projection(pooled_output)

# ─────────────────────────────────────────
# 11. Training Config
# ─────────────────────────────────────────
SEEDS       = [42, 601, 606, 619, 623]
EPOCHS      = 20
LR          = 5e-6
TEMPERATURE = 0.05
BATCH_SIZE  = 128

all_seed_losses = []

# ─────────────────────────────────────────
# 12. Load Processor + DataLoader once
#     From cache — no network call
# ─────────────────────────────────────────
print("\nLoading processor from cache...")
processor = CLIPProcessor.from_pretrained(CLIP_CACHE)
print("✅ Processor loaded.")

dataset    = DeepFashionContrastiveDataset(df_train, cropped_dir, processor)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)
print(f"DataLoader ready: {len(dataloader)} batches/epoch\n")

# ─────────────────────────────────────────
# 13. Training Loop
# ─────────────────────────────────────────
final_zip_path = f'{final_zip}.zip'

for seed in SEEDS:
    print(f"\n{'='*55}")
    print(f"Training seed : {seed}")
    print(f"Epochs        : {EPOCHS}")
    print(f"Batch size    : {BATCH_SIZE}")
    print(f"Temperature   : {TEMPERATURE}")
    print(f"LR            : {LR}")
    print(f"Disk used     : {get_dir_size_gb('/kaggle/working'):.1f} GB")
    print(f"{'='*55}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # ✅ Load from local cache — instant, no network
    print("Loading CLIP from local cache...")
    clip_model = CLIPModel.from_pretrained(CLIP_CACHE).to(DEVICE)
    print("✅ CLIP loaded from disk.")

    # Freeze everything
    for param in clip_model.parameters():
        param.requires_grad = False

    # Unfreeze ALL vision encoder blocks
    vision_encoder = clip_model.vision_model
    total_blocks   = len(vision_encoder.encoder.layers)
    print(f"Unfreezing all {total_blocks} vision blocks + embeddings")

    for layer in vision_encoder.encoder.layers:
        for param in layer.parameters():
            param.requires_grad = True

    for param in vision_encoder.embeddings.parameters():
        param.requires_grad = True

    for param in vision_encoder.post_layernorm.parameters():
        param.requires_grad = True

    for param in clip_model.visual_projection.parameters():
        param.requires_grad = True

    # Text encoder stays frozen per spec
    for param in clip_model.text_model.parameters():
        param.requires_grad = False
    for param in clip_model.text_projection.parameters():
        param.requires_grad = False

    trainable = sum(p.numel() for p in clip_model.parameters()
                    if p.requires_grad)
    total_p   = sum(p.numel() for p in clip_model.parameters())
    print(f"Trainable: {trainable:,} / {total_p:,} "
          f"({100*trainable/total_p:.1f}%)")

    if torch.cuda.device_count() > 1:
        clip_model = nn.DataParallel(clip_model)
        print("DataParallel across both GPUs")

    criterion = NTXentLoss(temperature=TEMPERATURE).to(DEVICE)

    # Layer-wise learning rates
    m = clip_model.module if isinstance(clip_model, nn.DataParallel) \
        else clip_model

    param_groups = [
        # Embeddings — smallest LR
        {'params': m.vision_model.embeddings.parameters(),
         'lr'    : LR * 0.1},
        # Early blocks (0-5) — half LR
        {'params': [p for i, layer in
                    enumerate(m.vision_model.encoder.layers)
                    if i < 6
                    for p in layer.parameters()],
         'lr'    : LR * 0.5},
        # Later blocks (6-11) — full LR
        {'params': [p for i, layer in
                    enumerate(m.vision_model.encoder.layers)
                    if i >= 6
                    for p in layer.parameters()],
         'lr'    : LR},
        # Post layernorm + projection — full LR
        {'params': list(m.vision_model.post_layernorm.parameters()) +
                   list(m.visual_projection.parameters()),
         'lr'    : LR},
    ]

    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

    # Warmup + Cosine schedule
    total_steps  = EPOCHS * len(dataloader)
    warmup_steps = len(dataloader) * 2

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / \
                   max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_loss = float('inf')
    seed_best = os.path.join(temp_dir, f'seed_{seed}', 'best')
    os.makedirs(seed_best, exist_ok=True)

    for epoch in range(EPOCHS):
        clip_model.train()
        epoch_loss  = 0.0
        num_batches = 0
        epoch_start = time.time()

        pbar = tqdm(dataloader,
                    desc=f"Seed {seed} | Epoch {epoch+1}/{EPOCHS}")

        for batch in pbar:
            pv1 = batch["pixel_values_1"].to(DEVICE)
            pv2 = batch["pixel_values_2"].to(DEVICE)

            z1 = F.normalize(get_vision_features(clip_model, pv1), dim=-1)
            z2 = F.normalize(get_vision_features(clip_model, pv2), dim=-1)

            loss = criterion(z1, z2)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                clip_model.parameters(), max_norm=1.0
            )
            optimizer.step()
            scheduler.step()

            epoch_loss  += loss.item()
            num_batches += 1

            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'avg' : f"{epoch_loss/num_batches:.4f}",
                'lr'  : f"{scheduler.get_last_lr()[0]:.2e}"
            })

        avg_loss   = epoch_loss / num_batches
        epoch_time = time.time() - epoch_start

        print(f"Seed {seed} | Epoch {epoch+1}/{EPOCHS} "
              f"— loss: {avg_loss:.4f} "
              f"— time: {epoch_time/60:.1f} mins")

        # Save only best checkpoint
        if avg_loss < best_loss:
            best_loss     = avg_loss
            model_to_save = (clip_model.module
                             if isinstance(clip_model, nn.DataParallel)
                             else clip_model)
            model_to_save.save_pretrained(seed_best)
            processor.save_pretrained(seed_best)
            print(f"  ✅ Best saved (loss: {best_loss:.4f})")

    all_seed_losses.append(best_loss)
    print(f"\nSeed {seed} complete — best loss: {best_loss:.4f}")

    # Free GPU memory
    del clip_model
    gc.collect()
    torch.cuda.empty_cache()

    # ── Add seed to single zip → delete folder ──
    print(f"  Adding seed_{seed} to zip...")
    with zipfile.ZipFile(final_zip_path, 'a',
                         compression=zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(seed_best):
            for file in files:
                file_path = os.path.join(root, file)
                arc_name  = os.path.join(
                    'clip_finetuned_v2',
                    f'seed_{seed}',
                    'best',
                    file
                )
                zf.write(file_path, arc_name)

    zip_size = os.path.getsize(final_zip_path) / 1e6
    print(f"  ✅ Zip updated: {zip_size:.0f} MB")

    # Delete temp folder to free disk
    shutil.rmtree(os.path.join(temp_dir, f'seed_{seed}'))
    print(f"  🗑️  Deleted temp/seed_{seed}")
    print(f"  💾 Disk used: {get_dir_size_gb('/kaggle/working'):.1f} GB")

# ─────────────────────────────────────────
# 14. Final Summary
# ─────────────────────────────────────────
losses_arr = np.array(all_seed_losses)
print(f"\n{'='*55}")
print(f"Results across seeds : {SEEDS}")
print(f"Losses               : {[f'{l:.4f}' for l in all_seed_losses]}")
print(f"Mean loss            : {losses_arr.mean():.4f}")
print(f"Std loss             : {losses_arr.std():.4f}")
print(f"Final                : {losses_arr.mean():.4f} ± {losses_arr.std():.4f}")
print(f"{'='*55}")

best_seed = SEEDS[int(np.argmin(all_seed_losses))]
print(f"\nBest seed       : {best_seed}")
print(f"Final zip       : {final_zip_path}")
print(f"Final zip size  : {os.path.getsize(final_zip_path)/1e6:.0f} MB")
print(f"Disk used       : {get_dir_size_gb('/kaggle/working'):.1f} GB")

# Cleanup
shutil.rmtree(temp_dir, ignore_errors=True)
print("\n✅ Done! Download clip_finetuned_v2.zip from Kaggle output.")

GPUs available: 2
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HuggingFace login successful.



config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

  ✅ Best saved (loss: 0.8236)


Seed 623 | Epoch 19/20: 100%|██████████| 31/31 [01:12<00:00,  2.33s/it, loss=0.7710, avg=0.8505, lr=3.80e-09]


Seed 623 | Epoch 19/20 — loss: 0.8505 — time: 1.2 mins


Seed 623 | Epoch 20/20: 100%|██████████| 31/31 [01:11<00:00,  2.32s/it, loss=0.9174, avg=0.8680, lr=0.00e+00]


Seed 623 | Epoch 20/20 — loss: 0.8680 — time: 1.2 mins

Seed 623 complete — best loss: 0.8236
  Adding seed_623 to zip...
  ✅ Zip updated: 2553 MB
  🗑️  Deleted temp/seed_623
  💾 Disk used: 3.2 GB

Results across seeds : [42, 601, 606, 619, 623]
Losses               : ['0.8222', '0.8478', '0.8273', '0.8269', '0.8236']
Mean loss            : 0.8296
Std loss             : 0.0093
Final                : 0.8296 ± 0.0093

Best seed       : 42
Final zip       : /kaggle/working/clip_finetuned_v2.zip
Final zip size  : 2553 MB
Disk used       : 3.2 GB

✅ Done! Download clip_finetuned_v2.zip from Kaggle output.
